In [ ]:
!pip install -q git+https://github.com/huggingface/transformers.git

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import random
import requests
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import nltk
from nltk.corpus import wordnet
import warnings
warnings.filterwarnings('ignore')

try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
except OSError:
    os.system("python -m spacy download en_core_web_sm")
    import spacy
    nlp = spacy.load("en_core_web_sm")

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device in use: {device}\n")

BASE_INPUT_PATH = "/kaggle/input/datasets/claudiacornacchia/"
EMBEDDING_DIR = "embedding"
csv_path = os.path.join(BASE_INPUT_PATH, EMBEDDING_DIR, "base_map.csv")

emb_model_id = "Qwen/Qwen3-Embedding-0.6B"
print(f"Loading {emb_model_id}...")
emb_tokenizer = AutoTokenizer.from_pretrained(emb_model_id, trust_remote_code=True)
emb_tokenizer.padding_side = "right"          
emb_model = AutoModel.from_pretrained(emb_model_id, trust_remote_code=True).to(device)
emb_model.eval()

gen_model_id = "google/gemma-4-E2B-it"
print(f"Loading {gen_model_id} (Requires high VRAM)...")
gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_id)
gen_model = AutoModelForCausalLM.from_pretrained(
    gen_model_id, 
    torch_dtype=torch.bfloat16,
    device_map="cuda:0" if torch.cuda.is_available() else "cpu"  
)
print("Models loaded successfully!\n")

def get_embedding(text):
    inputs = emb_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = emb_model(**inputs)
        attention_mask = inputs['attention_mask']
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        embedding = sum_embeddings / sum_mask
        embedding = F.normalize(embedding, p=2, dim=1)
    return embedding

def get_wordnet_pos_from_spacy(spacy_pos):
    if spacy_pos == 'ADJ': return wordnet.ADJ
    elif spacy_pos == 'VERB': return wordnet.VERB
    elif spacy_pos == 'NOUN': return wordnet.NOUN
    elif spacy_pos == 'ADV': return wordnet.ADV
    return None

def get_single_word_replacement(spacy_token, mode="synonym"):
    wn_tag = get_wordnet_pos_from_spacy(spacy_token.pos_)
    replacements = set()
    if wn_tag:
        for syn in wordnet.synsets(spacy_token.lemma_, pos=wn_tag):
            for lemma in syn.lemmas():
                if mode == "synonym" and lemma.name().lower() != spacy_token.lemma_.lower():
                    replacements.add(lemma.name().replace('_', ' '))
                elif mode == "antonym" and lemma.antonyms():
                    replacements.add(lemma.antonyms()[0].name().replace('_', ' '))

    if not replacements:
        try:
            word = spacy_token.text.lower()
            url = f"https://api.datamuse.com/words?ml={word}&max=5" if mode == "synonym" else f"https://api.datamuse.com/words?rel_ant={word}&max=5"
            response = requests.get(url, timeout=2).json()
            for item in response:
                if " " not in item['word']:
                    replacements.add(item['word'])
        except:
            pass
    if replacements:
        new_word = random.choice(list(replacements))
        if spacy_token.is_title: new_word = new_word.title()
        elif spacy_token.is_upper: new_word = new_word.upper()
        return new_word
    return None

def perturb_dict_safe(text, mode="synonym"):
    doc = nlp(text)
    new_text = ""
    changed = False
    for token in doc:
        if not changed and token.is_alpha:
            replacement = get_single_word_replacement(token, mode=mode)
            if replacement:
                new_text += replacement + token.whitespace_
                changed = True
                continue
        new_text += token.text_with_ws
    return new_text if changed else text

def perturb_typo(text):
    doc = nlp(text)
    tokens = list(doc)
    candidates = [i for i, t in enumerate(tokens) if len(t.text) > 4 and t.is_alpha]
    if not candidates: return text
    target_idx = random.choice(candidates)
    word = tokens[target_idx].text
    swap_idx = random.randint(1, len(word) - 3)
    new_word = word[:swap_idx] + word[swap_idx+1] + word[swap_idx] + word[swap_idx+2:]
    new_text = ""
    for i, t in enumerate(tokens):
        if i == target_idx: new_text += new_word + t.whitespace_
        else: new_text += t.text_with_ws
    return new_text

def perturb_noise(text):
    doc = nlp(text)
    if len(doc) == 0: return text
    noise_words = ["uh", "um", "like", "literally", "basically", "actually", "lol", "lmao", "smh", "btw"]
    tokens_with_ws = [token.text_with_ws for token in doc]
    num_noise = random.randint(1, 2)
    for _ in range(num_noise):
        insert_idx = random.randint(0, len(tokens_with_ws))
        tokens_with_ws.insert(insert_idx, random.choice(noise_words) + " ")
    return "".join(tokens_with_ws)

def perturb_shuffle(text):
    doc = nlp(text)
    word_tokens = [t for t in doc if t.is_alpha]
    if len(word_tokens) <= 2: return text
    shuffled_texts = [t.text for t in word_tokens]
    random.shuffle(shuffled_texts)
    new_text = ""
    word_idx = 0
    for t in doc:
        if t.is_alpha:
            new_text += shuffled_texts[word_idx] + t.whitespace_
            word_idx += 1
        else:
            new_text += t.text_with_ws
    return new_text

def perturb_leetspeak(text):
    doc = nlp(text)
    leet_map = {'a':'4', 'e':'3', 'i':'1', 'o':'0', 's':'$', 't':'7'}
    new_text = ""
    changed = False
    for t in doc:
        if t.is_alpha and len(t.text) > 3:
            word = ""
            for char in t.text:
                if char.lower() in leet_map and random.random() > 0.4:
                    word += leet_map[char.lower()]
                    changed = True
                else: word += char
            new_text += word + t.whitespace_
        else: new_text += t.text_with_ws
    return new_text if changed else text

def perturb_llm_safe(text, mode="paraphrase"):
    instruction = ("Rewrite the following airline review tweet using completely different words, "
                   "but keep the exact same meaning and sentiment." if mode == "paraphrase"
                   else "Rewrite the following airline review tweet to have the exact opposite sentiment. If it complains, make it praise the airline. If it praises, make it complain.")
    messages = [
        {"role": "system", "content": "You are a helpful assistant specialized in rewriting text. Output ONLY the rewritten text, without explanations, conversational filler, markdown blocks, or quotes."},
        {"role": "user", "content": f"{instruction}\n\nTweet: {text}"}
    ]
    text_input = gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = gen_tokenizer([text_input], return_tensors="pt").to(gen_model.device)    
    with torch.no_grad():
        generated_ids = gen_model.generate(
            **model_inputs, 
            max_new_tokens=100,        
            temperature=0.7,
            top_p=0.95,                
            top_k=64,                  
            do_sample=True
        )
    output_ids = generated_ids[0][model_inputs.input_ids.shape[1]:]
    result = gen_tokenizer.decode(output_ids, skip_special_tokens=True).strip()
    torch.cuda.empty_cache()
    return result

def run_stress_test(df_sample):
    results = []
    print(f"\n=== STARTING STRESS-TEST ON {len(df_sample)} TWEETS ===")
    print("Applying 8 different perturbations to each tweet.\n")
    
    for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Perturbing Tweets"):
        orig_text = str(row['text'])
        orig_emb = get_embedding(orig_text)
        
        # 1. Apply all attacks
        t_syn = perturb_dict_safe(orig_text, "synonym")
        t_ant = perturb_dict_safe(orig_text, "antonym")
        t_typ = perturb_typo(orig_text)
        t_noi = perturb_noise(orig_text)
        t_shf = perturb_shuffle(orig_text)
        t_let = perturb_leetspeak(orig_text)
        t_par = perturb_llm_safe(orig_text, "paraphrase")
        t_opp = perturb_llm_safe(orig_text, "opposite")
        
        emb_syn = get_embedding(t_syn) if t_syn != orig_text else orig_emb
        emb_ant = get_embedding(t_ant) if t_ant != orig_text else orig_emb
        emb_typ = get_embedding(t_typ) if t_typ != orig_text else orig_emb
        emb_noi = get_embedding(t_noi) if t_noi != orig_text else orig_emb
        emb_shf = get_embedding(t_shf) if t_shf != orig_text else orig_emb
        emb_let = get_embedding(t_let) if t_let != orig_text else orig_emb
        emb_par = get_embedding(t_par) if t_par != orig_text else orig_emb
        emb_opp = get_embedding(t_opp) if t_opp != orig_text else orig_emb
        
        results.append({
            'tweet_id': row.get('tweet_id', idx),
            'Airline': row.get('airline', 'Unknown'),
            'Original_Text': orig_text,
            'Synonym_Impact': 1.0 - F.cosine_similarity(orig_emb, emb_syn).item(),
            'Antonym_Impact': 1.0 - F.cosine_similarity(orig_emb, emb_ant).item(),
            'Typo_Impact': 1.0 - F.cosine_similarity(orig_emb, emb_typ).item(),
            'Noise_Impact': 1.0 - F.cosine_similarity(orig_emb, emb_noi).item(),
            'Shuffle_Impact': 1.0 - F.cosine_similarity(orig_emb, emb_shf).item(),
            'Leetspeak_Impact': 1.0 - F.cosine_similarity(orig_emb, emb_let).item(),
            'LLM_Paraphrase_Impact': 1.0 - F.cosine_similarity(orig_emb, emb_par).item(),
            'LLM_Opposite_Impact': 1.0 - F.cosine_similarity(orig_emb, emb_opp).item()
        })
        
        torch.cuda.empty_cache()
        
    df_results = pd.DataFrame(results)
    
    print("\n" + "="*50)
    print(" QWEN3 VULNERABILITY RANKING ")
    print("="*50)
    print("(Higher values indicate greater embedding distortion caused by the attack)\n")
    
    mean_impacts = df_results[[
        'Synonym_Impact', 'Antonym_Impact', 'Typo_Impact', 'Noise_Impact', 
        'Shuffle_Impact', 'Leetspeak_Impact', 'LLM_Paraphrase_Impact', 'LLM_Opposite_Impact'
    ]].mean().sort_values(ascending=False)
    
    for attack, impact in mean_impacts.items():
        attack_name = attack.replace("_Impact", "")
        print(f" {attack_name:<20} Average Impact: {impact:.4f}")
        
    return df_results

if __name__ == "__main__":
    try:
        df_dati = pd.read_csv(csv_path)
        # safe_sample = 50  
        # df_test = df_dati.sample(safe_sample, random_state=42)
        
        # df_benchmark = run_stress_test(df_test)
        df_benchmark = run_stress_test(df_dati)
        
        output_file = "/kaggle/working/stress_test_benchmark.csv"
        df_benchmark.to_csv(output_file, index=False)
        print(f"\n Analysis complete! Results saved to: {output_file}")
        
    except FileNotFoundError:
        print(f"ERROR: File {csv_path} not found. Please ensure the dataset is in the correct path.")